# FAIREA Analysis from Pre-computed Model Results

## Introduction

This notebook demonstrates how to apply the **FAIREA (Fairness Area)** methodology using a pre-existing DataFrame of model performance metrics. Unlike the original FAIREA examples which generate a baseline by mutating a single model's predictions, this notebook constructs the baseline from a collection of existing 'baseline' model results.

The primary objective is to replicate the FAIREA analysis and visualization workflow to evaluate various bias mitigation techniques. This includes:
1.  Modeling an accuracy-vs-fairness trade-off curve using a **linear regression** of the provided baseline models.
2.  Normalizing the performance metrics of both baseline and bias-mitigated models against this curve.
3.  Classifying each mitigation method into a specific trade-off region (e.g., "Win-Win", "Good Trade-off", "Lose-Lose").
4.  Quantifying and visualizing the area of improvement for methods in the "Good Trade-off" region.

This approach allows us to leverage existing experimental results without running new classification tasks, making it ideal for meta-analysis and model comparison.

## 1. Import Packages

In [2]:
# Standard library imports for path management and warnings
import sys
import warnings

# Append parent directory to path to locate the Fairea module if needed
sys.path.append("../")
warnings.filterwarnings('ignore')

# Data manipulation and numerical operations
import numpy as np
import pandas as pd

# Scikit-learn for linear regression
from sklearn.linear_model import LinearRegression

# FAIREA-specific functions for analysis from the provided library
from Fairea.fairea import normalize, classify_region

# Plotting and visualization libraries
from matplotlib import pyplot as plt
import plotly.graph_objects as go
import matplotlib

# Geometric objects for area calculation
from shapely.geometry import Polygon, Point, LineString

# Matplotlib configuration for consistent plot styling
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rcParams.update({'font.size': 12})

***
**Validation:** All necessary packages, including `sklearn.linear_model`, were imported successfully.
***

## 2. Data Loading and Preprocessing

We load the aggregated model results from a CSV file. The DataFrame is then validated to ensure it contains the required columns (`BM`, `balanced_acc`, `fair_score`, `model_clean`). We then split the data into two groups:

-   **Baseline models:** Rows where `BM` is 'baseline'. These points will be used to train our linear regression model for the trade-off curve.
-   **Mitigation methods:** All other rows, representing techniques to be evaluated.


In [3]:
# Define the path to the data file
data_path = '../../../experiments_intersec/stroke_age_ever_married_Residence_type_agg.csv'

try:
    # Load data from the specified CSV file
    df_results = pd.read_csv(data_path)
    
    # --- Data Validation ---
    required_columns = ['BM', 'balanced_acc', 'fair_score', 'model_clean']
    if not all(col in df_results.columns for col in required_columns):
        missing = set(required_columns) - set(df_results.columns)
        raise ValueError(f"DataFrame is missing required columns: {missing}")
    
    if 'baseline' not in df_results['BM'].unique():
        raise ValueError("No 'baseline' models found in 'BM' column. Cannot generate a tradeoff curve.")

    # --- Data Preprocessing ---
    df_results['fairness'] = df_results['fair_score']
    baseline_df = df_results[df_results['BM'] == 'baseline'].copy()
    methods_df = df_results[df_results['BM'] != 'baseline'].copy()
    
    # Sort baseline models by fairness for consistent plotting of raw points
    baseline_df = baseline_df.sort_values(by='fairness').reset_index(drop=True)
    
    print("Data loaded and preprocessed successfully.")
    print(f"Found {len(baseline_df)} baseline models and {len(methods_df)} mitigation methods.")
    
except FileNotFoundError:
    print(f"Error: The data file was not found at '{data_path}'. Please check the path.")
except ValueError as e:
    print(f"Data Validation Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Data loaded and preprocessed successfully.
Found 8 baseline models and 97 mitigation methods.


In [4]:
df_results.head()

,model_clean,BM,balanced_acc,acc,precision,recall,f1,mcc,eq_opp_diff,avg_odd_diff,spd,disparate_impact,theil_idx,fair_score,model_type,fairness
0,LogisticRegression,baseline,0.678049,0.735450,0.272727,0.60,0.375000,0.131029,-0.50,-0.657895,-0.779070,0.220930,0.105880,1.0,linear,1.0
1,XGBClassifier,baseline,0.595732,0.592593,0.182927,0.60,0.280374,0.110185,-0.50,-0.569079,-0.622093,0.377907,0.113983,1.0,tree_boost,1.0
2,RandomForestClassifier,baseline,0.630976,0.624339,0.205128,0.64,0.310680,0.113164,-0.45,-0.515570,-0.580711,0.382994,0.107976,1.0,tree_bag,1.0
3,CatBoostClassifier,baseline,0.668902,0.719577,0.258621,0.60,0.361446,0.094431,-0.50,-0.648026,-0.761628,0.238372,0.107586,1.0,tree_boost,1.0
4,TabNetClassifier,baseline,0.653659,0.693122,0.238095,0.60,0.340909,0.122710,-0.50,-0.541667,-0.603283,0.316279,0.109934,1.0,tabnet,1.0


## 3. Baseline Modeling and Data Preparation

Instead of connecting the raw baseline points to form the trade-off curve, we will model the general trend using a **linear regression**. This provides a smoother, more generalized representation of the inherent trade-off between accuracy and fairness for the baseline models. We define a function `get_regression_diagonal` to handle this calculation.

In [5]:
import numpy as np
from sklearn.linear_model import LinearRegression

def get_regression_diagonal_with_threshold(baseline_df, method='accuracy_threshold', 
                                          accuracy_target=None, percentile=85, range_fraction=0.67):
    """
    Calculates the start and end coordinates of a linear regression line
    based on the baseline model performance.
    
    Args:
        baseline_df (pd.DataFrame): DataFrame containing 'fairness' and 'balanced_acc' columns
        method (str): Method to determine diagonal endpoint:
            - 'mean': End at mean of fairness values (simple, robust)
            - 'harmonic_mean': End at harmonic mean (heavily weights low values)
            - 'weighted_mean': End at 60% of range (compromise between min and max)
            - 'range_fraction': End at fraction of fairness range (e.g., 2/3)
            - 'accuracy_threshold': End at point where regression crosses target accuracy
            - 'percentile': End at specified percentile of fairness
            - 'mean_plus_2_sqrt': End at mean(fairness) + 2 * sqrt(Var(fairness)), clipped to range
        accuracy_target (float): Target accuracy for 'accuracy_threshold' method
        percentile (int): Percentile for 'percentile' method (default: 85)
        range_fraction (float): Fraction of range for 'range_fraction' method (default: 0.67)
    
    Returns:
        tuple: ((x1, y1), (x2, y2)) - start and end points of diagonal
    """
    # Prepare the data for scikit-learn
    X = baseline_df[['fairness']]
    y = baseline_df['balanced_acc']
    
    # Create and fit the linear regression model
    model = LinearRegression()
    model.fit(X, y)
    
    # Start point: minimum fairness value
    x1 = baseline_df['fairness'].min()
    y1 = model.predict([[x1]])[0]
    
    # Get fairness range
    fair_min = baseline_df['fairness'].min()
    fair_max = baseline_df['fairness'].max()
    fair_range = fair_max - fair_min
    
    # Determine end point based on method
    if method == 'mean':
        x2 = baseline_df['fairness'].mean()
        y2 = model.predict([[x2]])[0]
        
    elif method == 'harmonic_mean':
        fairness_values = baseline_df['fairness'].values
        fairness_safe = np.where(fairness_values == 0, 0.001, fairness_values)
        x2 = len(fairness_safe) / np.sum(1.0 / fairness_safe)
        y2 = model.predict([[x2]])[0]
        
    elif method == 'weighted_mean':
        x2 = 0.4 * fair_min + 0.6 * fair_max
        y2 = model.predict([[x2]])[0]
    
    elif method == 'range_fraction':
        x2 = fair_min + (fair_range * range_fraction)
        y2 = model.predict([[x2]])[0]
        
    elif method == 'accuracy_threshold':
        if accuracy_target is None:
            accuracy_target = np.percentile(baseline_df['balanced_acc'], 75)
        slope = model.coef_[0]
        intercept = model.intercept_
        if slope != 0:
            x2 = (accuracy_target - intercept) / slope
            x2 = np.clip(x2, fair_min, fair_max)
        else:
            x2 = baseline_df['fairness'].mean()
        y2 = model.predict([[x2]])[0]
        
    elif method == 'percentile':
        x2 = np.percentile(baseline_df['fairness'], percentile)
        y2 = model.predict([[x2]])[0]

    elif method == 'mean_plus_2_sqrt':
        # Compute mean and (population) variance of fairness
        mu = float(baseline_df['fairness'].mean())
        var = float(baseline_df['fairness'].var(ddof=0))  # population variance
        # Guard against tiny negative due to numerical precision
        var = max(var, 0.0)
        x2_raw = mu + 2.0 * np.sqrt(var)
        # Clip to observed range to stay inside the data domain
        x2 = float(np.clip(x2_raw, fair_min, fair_max))
        y2 = model.predict([[x2]])[0]
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return (x1, y1), (x2, y2)


def visualize_regression_diagonal(baseline_df, title="Regression Diagonal Visualization"):
    """
    Visualizes the regression diagonal on a 2D plot of balanced_acc vs fair_score.
    
    Args:
        baseline_df (pd.DataFrame): DataFrame with 'fairness' and 'balanced_acc' columns
        title (str): Plot title
    
    Returns:
        plotly.graph_objects.Figure: Interactive plotly figure
    """
    # Get the diagonal coordinates
    (x1, y1), (x2, y2) = get_regression_diagonal_with_threshold(baseline_df, method='accuracy_threshold')
    
    # Fit the full regression line for visualization
    X = baseline_df[['fairness']]
    y = baseline_df['balanced_acc']
    model = LinearRegression()
    model.fit(X, y)
    
    # Create points for the full regression line
    x_range = np.linspace(baseline_df['fairness'].min(), baseline_df['fairness'].max(), 100)
    y_pred = model.predict(x_range.reshape(-1, 1))
    
    # Find min and max accuracy points
    min_acc_idx = y.idxmin()
    max_acc_idx = y.idxmax()
    
    fig = go.Figure()
    
    
    # Add the diagonal (red, thick)
    fig.add_trace(go.Scatter(
        x=[x1, x2],
        y=[y1, y2],
        mode='lines+markers',
        line=dict(color='red', width=4),
        marker=dict(size=12, color='red', symbol='diamond'),
        name=f'Diagonal<br>Slope: {model.coef_[0]:.4f}',
        showlegend=True,
        hovertemplate='<b>Diagonal Point</b><br>Fairness: %{x:.3f}<br>Balanced Acc: %{y:.3f}<extra></extra>'
    ))
    
    # Add all baseline points
    fig.add_trace(go.Scatter(
        x=baseline_df['fairness'],
        y=baseline_df['balanced_acc'],
        mode='markers',
        marker=dict(size=10, color='blue', opacity=0.6),
        name='Baseline Models',
        text=baseline_df.index,
        hovertemplate='<b>Model: %{text}</b><br>Fairness: %{x:.3f}<br>Balanced Acc: %{y:.3f}<extra></extra>'
    ))
    
    # Highlight min accuracy point
    fig.add_trace(go.Scatter(
        x=[baseline_df.loc[min_acc_idx, 'fairness']],
        y=[baseline_df.loc[min_acc_idx, 'balanced_acc']],
        mode='markers',
        marker=dict(size=15, color='orange', symbol='star', line=dict(width=2, color='darkslategrey')),
        name=f'Min Accuracy Point<br>x={x1:.3f}',
        showlegend=True,
        hovertemplate='<b>Min Accuracy</b><br>Fairness: %{x:.3f}<br>Balanced Acc: %{y:.3f}<extra></extra>'
    ))
    
    # Highlight max accuracy point
    fig.add_trace(go.Scatter(
        x=[baseline_df.loc[max_acc_idx, 'fairness']],
        y=[baseline_df.loc[max_acc_idx, 'balanced_acc']],
        mode='markers',
        marker=dict(size=15, color='green', symbol='star', line=dict(width=2, color='darkslategrey')),
        name=f'Max Accuracy Point<br>x={x2:.3f}',
        showlegend=True,
        hovertemplate='<b>Max Accuracy</b><br>Fairness: %{x:.3f}<br>Balanced Acc: %{y:.3f}<extra></extra>'
    ))
    
    # Add annotations for the diagonal endpoints
    fig.add_annotation(
        x=x1, y=y1,
        text=f"Start<br>({x1:.3f}, {y1:.3f})",
        showarrow=True,
        arrowhead=2,
        arrowcolor="red",
        ax=-40, ay=-40,
        font=dict(color="red", size=10)
    )
    
    fig.add_annotation(
        x=x2, y=y2,
        text=f"End<br>({x2:.3f}, {y2:.3f})",
        showarrow=True,
        arrowhead=2,
        arrowcolor="red",
        ax=40, ay=40,
        font=dict(color="red", size=10)
    )
    
    # Calculate R² score
    r2_score = model.score(X, y)
    
    fig.update_layout(
        title=f"{title}<br><sub>R² = {r2_score:.4f}, Intercept = {model.intercept_:.4f}</sub>",
        xaxis_title="Fair Score (Fairness)",
        yaxis_title="Balanced Accuracy",
        plot_bgcolor='white',
        showlegend=True,
        hovermode='closest',
        width=900,
        height=700
    )
    
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

    return fig


fig = visualize_regression_diagonal(baseline_df)
fig.show()
# --- Prepare data for FAIREA analysis ---

# Extract raw baseline scores for normalization
base_accuracy = baseline_df['balanced_acc'].to_numpy()
base_fairness = baseline_df['fairness'].to_numpy()

# Create the dictionary of mitigation methods
methods = {
    row['model_clean']: (row['balanced_acc'], row['fairness']) 
    for _, row in methods_df.iterrows()
}

***
**Validation:** The `get_regression_diagonal` function has been defined, and the data structures required for the FAIREA analysis are prepared.
***

## 4. FAIREA Analysis and Visualization

We now apply the FAIREA methodology. The performance scores are normalized, methods are classified into regions, and the "good trade-off" areas are calculated. The final plot will now show the linear regression line as the baseline diagonal, providing a clear visualization of the trade-off regions.

In [6]:
# Step 1: Normalize the baseline and method scores
normalized_accuracy, normalized_fairness, normalized_methods = normalize(base_accuracy, base_fairness, methods)

# Step 2: Create a geometric LineString from the normalized baseline for intersection calculations
# Note: We still use the original point-to-point baseline for area calculations as per FAIREA methodology.
baseline_geom = LineString(list(zip(normalized_fairness, normalized_accuracy)))

# Step 3: Classify each mitigation method into its corresponding FAIREA region
regions = classify_region(baseline_geom, normalized_methods)

print("--- FAIREA Region Classification ---")
for method, region in regions.items():
    print(f"{method}: {region}")

--- FAIREA Region Classification ---
LogisticRegression: good trade-off
XGBClassifier: bad trade-off
RandomForestClassifier: win-win
CatBoostClassifier: good trade-off
TabNetClassifier: good trade-off
MLPClassifier: bad trade-off
AdversarialDebiasing: good trade-off
MetaFairClassifier: bad trade-off


***
**Validation:** Normalization and region classification steps were executed successfully. Each mitigation method has been assigned to a trade-off region.
***

### 4.1 Plot Normalized Trade-off and FAIREA Regions

This plot is the main output of the FAIREA methodology. It visualizes the performance space divided into five distinct trade-off regions: **Win-Win**, **Good Trade-off**, **Poor Trade-off**, and **Lose-Lose**. The regions are defined relative to the performance of the best-accuracy baseline model. The trade-off curve is now represented by the **linear regression** of the baseline models, shown as a dashed red line.

In [7]:
import numpy as np
import plotly.graph_objects as go

def plot_tradeoff(df, baseline_df, diagonal_method='accuracy_threshold', 
                 accuracy_target=None, diagonal_percentile=85, range_fraction=0.67,
                 region_colors=None, title="Accuracy vs Bias Trade-off Regions"):
    # Calculate dynamic ranges with 5% padding
    x_min = max(0, df['fair_score'].min() - 0.05)
    x_max = min(1, df['fair_score'].max() + 0.05)
    y_min = max(0, df['balanced_acc'].min() - 0.05)
    y_max = min(1, df['balanced_acc'].max() + 0.05)

    # Default colors (kept as in your code; uniform below-diagonal color)
    colors = {
        'win_win': 'rgba(0,200,0,0.1)',
        'good_tradeoff': 'rgba(100,200,100,0.1)',
        'poor_tradeoff': 'rgba(255,165,0,0.18)',  # slightly higher opacity for clarity
        'inverted': 'rgba(128,0,128,0.1)',
        'lose_lose': 'rgba(255,0,0,0.1)'
    }
    if region_colors:
        colors.update(region_colors)

    fig = go.Figure()

    # Get regression diagonal with chosen method
    diagonal_start, diagonal_end = get_regression_diagonal_with_threshold(
        baseline_df, 
        method=diagonal_method,
        accuracy_target=accuracy_target,
        percentile=diagonal_percentile,
        range_fraction=range_fraction
    )

    
    # Ensure diagonal goes from lower-left to upper-right
    if diagonal_start[1] > diagonal_end[1]:
        diagonal_start, diagonal_end = diagonal_end, diagonal_start
    
    diag_start_x = diagonal_start[0]
    diag_start_y = diagonal_start[1]
    diag_end_x = diagonal_end[0]
    diag_end_y = diagonal_end[1]

    # 1. Win-Win (upper-left RECTANGLE)
    fig.add_shape(type="rect",
        x0=x_min, y0=diag_end_y,
        x1=diag_end_x, y1=y_max,
        fillcolor=colors['win_win'], layer="below")

    # 2. Good Trade-off (upper triangle)
    fig.add_shape(type="path",
        path=f"M {diag_start_x} {diag_start_y} "
            f"L {diag_end_x} {diag_end_y} "
            f"L {diag_start_x} {diag_end_y} Z",
        fillcolor=colors['good_tradeoff'], layer="below")

    # 3+4. Poor Trade-off: fuse triangle and bottom-left rectangle into ONE polygon
    # The unified polygon covers from x_min to diag_end_x across y_min..diag_end_y and below the diagonal.
    # Path goes counter-clockwise around the whole below-diagonal envelope.
    fig.add_shape(
        type="path",
        path=(
            f"M {x_min} {y_min} "
            f"L {diag_end_x} {y_min} "
            f"L {diag_end_x} {diag_start_y} "
            f"L {diag_end_x} {diag_end_y} "  # right edge up to diagonal end y
            f"L {diag_start_x} {diag_start_y} "  # along diagonal to start point
            f"L {x_min} {diag_start_y} "  # left to border at start y
            f"L {x_min} {y_min} Z"        # down to bottom-left and close
        ),
        fillcolor=colors['poor_tradeoff'],
        layer="below"
    )

    # 5. Inverted (upper-right rectangle)
    fig.add_shape(type="rect", 
        x0=diag_end_x, y0=diag_start_y, 
        x1=x_max, y1=y_max,
        fillcolor=colors['inverted'], layer="below")

    # 6. Lose-Lose (lower-right rectangle)
    fig.add_shape(type="rect", 
        x0=diag_end_x, y0=y_min, 
        x1=x_max, y1=diag_start_y,
        fillcolor=colors['lose_lose'], layer="below")
    
    # Diagonal line
    fig.add_trace(go.Scatter(
            x=[diagonal_start[0], diagonal_end[0]],
            y=[diagonal_start[1], diagonal_end[1]],
            mode='lines',
            line=dict(color='red', width=4),
            name=f'Baseline',
            showlegend=True,
            hovertemplate='<b>Diagonal</b><br>Fair Score: %{x:.3f}<br>Balanced Accuracy: %{y:.3f}<extra></extra>'
        ))

    # Scatter points
    fig.add_trace(go.Scatter(
        x=df['fair_score'], 
        y=df['balanced_acc'],
        mode='markers', 
        marker=dict(size=10, color='blue'),
        text=df.apply(lambda row: f"{row['model_clean']}<br>BM: {row['BM']}", axis=1),
        hovertemplate='<b>%{text}</b><br>Fair Score: %{x:.3f}<br>Balanced Accuracy: %{y:.3f}<extra></extra>',
        name='Models'))

    # Annotations: positioned inside their regions
    annotations = [
        # Win-Win: center of the upper-left rectangle
        dict(
            x=(x_min + diag_end_x) / 2,
            y=diag_end_y + 0.5 * (y_max - diag_end_y),
            text="Win-Win", showarrow=False, font=dict(color='darkgreen', size=14)
        ),
        # Good Trade-off: place clearly inside the triangle,
        # closer to the left edge and away from the diagonal end.
        dict(
            x=diag_start_x + 0.22 * (diag_end_x - diag_start_x),   # moved left
            y=diag_start_y + 0.70 * (diag_end_y - diag_start_y),   # well inside
            text="Good Trade-off", showarrow=False, font=dict(color='green', size=14)
        ),
        # Poor Trade-off: below the diagonal, centered within unified polygon
        dict(
            x=(x_min + diag_end_x) / 2,
            y=diag_start_y + 0.25 * (diag_end_y - diag_start_y),
            text="Poor Trade-off", showarrow=False, font=dict(color='darkorange', size=14)
        ),
        # Inverted: center of its rectangle
        dict(
            x=(diag_end_x + x_max) / 2,
            y=(diag_start_y + y_max) / 2,
            text="Inverted", showarrow=False, font=dict(color='purple', size=14)
        ),
        # Lose-Lose: center of its rectangle
        dict(
            x=(diag_end_x + x_max) / 2,
            y=(y_min + diag_start_y) / 2,
            text="Lose-Lose", showarrow=False, font=dict(color='darkred', size=14)
        ),
    ]
    
    # Best fairness highlight (unchanged)
    min_fair_score_row = df.loc[df['fair_score'].idxmin()]
    fig.add_trace(go.Scatter(
        x=[min_fair_score_row['fair_score']],
        y=[min_fair_score_row['balanced_acc']],
        mode='markers',
        marker=dict(size=15, color='red', symbol='star', line=dict(width=2, color='DarkSlateGrey')),
        text=[f"{min_fair_score_row['model_clean']}<br>BM: {min_fair_score_row['BM']}"],
        hovertemplate='<b>Best Fairness</b><br>%{text}<br>Fair Score: %{x:.3f}<br>Balanced Accuracy: %{y:.3f}<extra></extra>',
        name=f"Best Fairness<br>Model: {min_fair_score_row['model_clean']}<br>BM: {min_fair_score_row['BM']}",
        showlegend=True
    ))

    # Best accuracy highlight (unchanged)
    max_acc_row = df.loc[df['balanced_acc'].idxmax()]
    fig.add_trace(go.Scatter(
        x=[max_acc_row['fair_score']],
        y=[max_acc_row['balanced_acc']],
        mode='markers',
        marker=dict(size=15, color='green', symbol='star', line=dict(width=2, color='DarkSlateGrey')),
        text=[f"{max_acc_row['model_clean']}<br>BM: {max_acc_row['BM']}"],
        hovertemplate='<b>Best Accuracy</b><br>%{text}<br>Fair Score: %{x:.3f}<br>Balanced Accuracy: %{y:.3f}<extra></extra>',
        name=f"Best Accuracy<br>Model: {max_acc_row['model_clean']}<br>BM: {max_acc_row['BM']}",
        showlegend=True
    ))

    fig.update_layout(
        title=f"{title}<br><sub>Baseline | x_end={diagonal_end[0]:.3f}, y_end={diagonal_end[1]:.3f}</sub>",
        xaxis_title="Fair Score",
        yaxis_title="Balanced Accuracy",
        xaxis=dict(range=[x_min - 0.05, x_max + 0.05]),
        yaxis=dict(range=[y_min - 0.05, y_max + 0.05]),
        annotations=annotations,
        showlegend=True,
        plot_bgcolor='white',
    )

    return fig


# fig = plot_tradeoff(df_results, baseline_df, diagonal_method='percentile', diagonal_percentile=12)
fig = plot_tradeoff(df_results, baseline_df, diagonal_method='mean', diagonal_percentile=10)
fig.show()


***
**Validation:** The final interactive FAIREA region plot has been successfully generated. The baseline is now correctly represented by a linear regression line, and all data points are plotted within the appropriate trade-off regions.
***

## Conclusion

This notebook successfully replicated the FAIREA analysis workflow using a pre-computed set of model results. Instead of generating a baseline through prediction mutation, we constructed a trade-off curve from existing baseline models and modeled its trend with a **linear regression**.

The analysis produced the following key insights:
- **Region Classification:** We classified each bias mitigation technique into its respective trade-off region relative to the performance of the best baseline model. The results show a mix of methods across "good trade-off," "lose-lose," and other categories, indicating that not all mitigation strategies provide a favorable balance between accuracy and fairness for this dataset.
- **Quantitative Area:** The FAIREA area score can still be computed relative to the original point-to-point baseline curve for methods identified as having a "good trade-off." This area metric quantifies the degree of improvement over the baseline, with a larger area indicating a more beneficial trade-off.

This adapted workflow demonstrates that the FAIREA methodology is flexible. It can be applied not only to raw prediction outputs but also to collections of pre-evaluated models, making it a valuable tool for meta-analysis and comparing model performance against a generalized trade-off trend.

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

def analyze_multi_attribute_trends(csv_files_dict, baseline_dfs_dict, 
                                   title="Fairness vs Accuracy: Multi-Attribute Trend Analysis"):
    """
    Analyzes and visualizes trends as more sensitive attributes are added.
    
    Args:
        csv_files_dict: Dictionary with any keys (e.g., 'age', 'age_gender', 'age_gender_race')
                       and values as DataFrames with 'fair_score', 'balanced_acc', 'BM', 'model_clean'
        baseline_dfs_dict: Dictionary with same keys and baseline DataFrames with 'fairness', 'balanced_acc'
        title: Plot title
    
    Returns:
        Plotly figure showing the trend
    """
    
    # Generate vibrant colors dynamically based on number of datasets
    n_datasets = len(csv_files_dict)
    
    # Use a vibrant color palette
    vibrant_colors = [
        'rgb(0, 123, 255)',      # Bright Blue
        'rgb(40, 167, 69)',      # Vibrant Green
        'rgb(220, 53, 69)',      # Bright Red
        'rgb(255, 193, 7)',      # Amber
        'rgb(156, 39, 176)',     # Purple
        'rgb(0, 188, 212)',      # Cyan
        'rgb(255, 87, 34)',      # Deep Orange
        'rgb(76, 175, 80)',      # Light Green
        'rgb(233, 30, 99)',      # Pink
        'rgb(121, 85, 72)',      # Brown
    ]
    
    # Create color mapping for each key
    colors_map = {}
    for idx, key in enumerate(sorted(csv_files_dict.keys())):
        colors_map[key] = {
            'color': vibrant_colors[idx % len(vibrant_colors)],
            'name': key.replace('_', ' ').title()  # Format name nicely
        }
    
    fig = go.Figure()
    
    # Store diagonal info for trend arrows
    diagonals = {}
    
    # Plot each attribute configuration
    for key in sorted(csv_files_dict.keys()):
        df = csv_files_dict[key]
        baseline_df = baseline_dfs_dict[key]
        
        color_info = colors_map[key]
        
        # Get regression diagonal
        diagonal_start, diagonal_end = get_regression_diagonal_with_threshold(baseline_df, method='mean')
        diagonals[key] = (diagonal_start, diagonal_end)
        
        # Plot diagonal
        fig.add_trace(go.Scatter(
            x=[diagonal_start[0], diagonal_end[0]],
            y=[diagonal_start[1], diagonal_end[1]],
            mode='lines',
            line=dict(color=color_info['color'], width=4, dash='dash'),
            name=f"Baseline {color_info['name']}",
            showlegend=True,
            legendgroup=key,
            hovertemplate=f"<b>Baseline {color_info['name']}</b><br>Fair Score: %{{x:.3f}}<br>Balanced Acc: %{{y:.3f}}<extra></extra>"
        ))
        
        # Plot scatter points
        fig.add_trace(go.Scatter(
            x=df['fair_score'],
            y=df['balanced_acc'],
            mode='markers',
            marker=dict(size=10, color=color_info['color'], opacity=0.7,
                       line=dict(width=1, color='white')),
            name=f"Models {color_info['name']}",
            legendgroup=key,
            text=df.apply(lambda row: f"{row['model_clean']}<br>BM: {row['BM']}", axis=1),
            hovertemplate=f"<b>{color_info['name']}</b><br>%{{text}}<br>Fair Score: %{{x:.3f}}<br>Balanced Acc: %{{y:.3f}}<extra></extra>"
        ))
    
    # Add trend arrows between baseline endpoints
    if len(diagonals) >= 2:
        sorted_keys = sorted(diagonals.keys())
        
        for i in range(len(sorted_keys) - 1):
            key1 = sorted_keys[i]
            key2 = sorted_keys[i + 1]
            
            end1 = diagonals[key1][1]  # endpoint of first diagonal
            end2 = diagonals[key2][1]  # endpoint of second diagonal
            
            # Add arrow from end1 to end2
            fig.add_annotation(
                x=end2[0], y=end2[1],
                ax=end1[0], ay=end1[1],
                xref='x', yref='y',
                axref='x', ayref='y',
                showarrow=True,
                arrowhead=3,
                arrowsize=1.5,
                arrowwidth=2,
                arrowcolor='red',
                opacity=0.7
            )
    
    # Calculate overall metrics trend
    metrics_summary = []
    for key in sorted(csv_files_dict.keys()):
        df = csv_files_dict[key]
        avg_fair = df['fair_score'].mean()
        avg_acc = df['balanced_acc'].mean()
        metrics_summary.append({
            'key': key,
            'avg_fair': avg_fair,
            'avg_acc': avg_acc,
            'name': colors_map.get(key, {'name': key})['name']
        })
    
    # Add annotation with trend summary
    summary_text = "<b>Average Metrics Trend:</b><br>"
    for m in metrics_summary:
        summary_text += f"{m['name']}: Fair={m['avg_fair']:.3f}, Acc={m['avg_acc']:.3f}<br>"
    
    fig.add_annotation(
        text=summary_text,
        xref="paper", yref="paper",
        x=0.02, y=0.98,
        showarrow=False,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=10),
        align="left",
        xanchor="left",
        yanchor="top"
    )
    
    fig.update_layout(
        title=title,
        xaxis_title="Fair Score (Lower = More Fair)",
        yaxis_title="Balanced Accuracy",
        showlegend=True,
        plot_bgcolor='white',
        hovermode='closest',
        width=1000,
        height=700
    )
    
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    
    return fig


def create_tradeoff_evolution_plot(csv_files_dict, baseline_dfs_dict,
                                   title="Trade-off Region Evolution Across Attributes"):
    """
    Creates subplots showing how trade-off regions evolve with more attributes.
    
    Args:
        csv_files_dict: Dictionary with keys like '1_attr', '2_attr', '3_attr'
        baseline_dfs_dict: Dictionary with same keys and baseline DataFrames
        title: Plot title
    
    Returns:
        Plotly figure with subplots
    """
    from plotly.subplots import make_subplots
    
    sorted_keys = sorted(csv_files_dict.keys())
    n_plots = len(sorted_keys)
    
    fig = make_subplots(
        rows=1, cols=n_plots,
        subplot_titles=[f"{i+1} Attribute(s)" for i in range(n_plots)],
        horizontal_spacing=0.08
    )
    
    colors = {
        'win_win': 'rgba(0,200,0,0.15)',
        'good_tradeoff': 'rgba(100,200,100,0.15)',
        'poor_tradeoff': 'rgba(255,165,0,0.15)',
        'inverted': 'rgba(128,0,128,0.15)',
        'lose_lose': 'rgba(255,0,0,0.15)'
    }
    
    for idx, key in enumerate(sorted_keys, 1):
        df = csv_files_dict[key]
        baseline_df = baseline_dfs_dict[key]
        
        # Get diagonal
        diagonal_start, diagonal_end = get_regression_diagonal_with_threshold(baseline_df, method='mean')
        
        if diagonal_start[1] > diagonal_end[1]:
            diagonal_start, diagonal_end = diagonal_end, diagonal_start
        
        diag_start_x, diag_start_y = diagonal_start
        diag_end_x, diag_end_y = diagonal_end
        
        # Calculate plot ranges
        x_min = max(0, df['fair_score'].min() - 0.05)
        x_max = min(1, df['fair_score'].max() + 0.05)
        y_min = max(0, df['balanced_acc'].min() - 0.05)
        y_max = min(1, df['balanced_acc'].max() + 0.05)
        
        # Add regions (simplified for subplots)
        # Win-Win
        fig.add_shape(type="rect",
            x0=x_min, y0=diag_end_y, x1=diag_end_x, y1=y_max,
            fillcolor=colors['win_win'], layer="below",
            row=1, col=idx)
        
        # Good Trade-off
        fig.add_shape(type="path",
            path=f"M {diag_start_x} {diag_start_y} L {diag_end_x} {diag_end_y} L {diag_start_x} {diag_end_y} Z",
            fillcolor=colors['good_tradeoff'], layer="below",
            row=1, col=idx)
        
        # Poor Trade-off (unified)
        fig.add_shape(type="path",
            path=f"M {x_min} {y_min} L {diag_end_x} {y_min} L {diag_end_x} {diag_start_y} "
                 f"L {diag_end_x} {diag_end_y} L {diag_start_x} {diag_start_y} "
                 f"L {x_min} {diag_start_y} L {x_min} {y_min} Z",
            fillcolor=colors['poor_tradeoff'], layer="below",
            row=1, col=idx)
        
        # Inverted
        fig.add_shape(type="rect",
            x0=diag_end_x, y0=diag_start_y, x1=x_max, y1=y_max,
            fillcolor=colors['inverted'], layer="below",
            row=1, col=idx)
        
        # Lose-Lose
        fig.add_shape(type="rect",
            x0=diag_end_x, y0=y_min, x1=x_max, y1=diag_start_y,
            fillcolor=colors['lose_lose'], layer="below",
            row=1, col=idx)
        
        # Add diagonal
        fig.add_trace(go.Scatter(
            x=[diag_start_x, diag_end_x],
            y=[diag_start_y, diag_end_y],
            mode='lines',
            line=dict(color='red', width=3),
            name=f'Baseline',
            showlegend=(idx == 1),
            legendgroup='baseline'
        ), row=1, col=idx)
        
        # Add scatter points
        fig.add_trace(go.Scatter(
            x=df['fair_score'],
            y=df['balanced_acc'],
            mode='markers',
            marker=dict(size=6, color='blue', opacity=0.6),
            name='Models',
            showlegend=(idx == 1),
            legendgroup='models',
            text=df.apply(lambda row: f"{row['model_clean']}<br>BM: {row['BM']}", axis=1),
            hovertemplate='<b>%{text}</b><br>Fair Score: %{x:.3f}<br>Balanced Acc: %{y:.3f}<extra></extra>'
        ), row=1, col=idx)
    
    fig.update_layout(
        title=title,
        showlegend=True,
        plot_bgcolor='white',
        height=500,
        width=1400
    )
    
    fig.update_xaxes(title_text="Fair Score", showgrid=True, gridcolor='lightgray')
    fig.update_yaxes(title_text="Balanced Accuracy", showgrid=True, gridcolor='lightgray')
    
    return fig


def load_and_prepare_data(data_path):
    """
    Loads data from CSV and prepares baseline and results dataframes.
    
    Args:
        data_path (str): Path to the CSV file containing the experimental results
        
    Returns:
        tuple: (baseline_df, results_df, full_df) where:
            - baseline_df: DataFrame with only baseline models
            - results_df: DataFrame with only bias mitigation methods
            - full_df: Complete DataFrame with all data
            
    Raises:
        FileNotFoundError: If the CSV file doesn't exist
        ValueError: If required columns are missing or no baseline models found
    """
    try:
        # Load data from the specified CSV file
        df_results = pd.read_csv(data_path)
        
        # --- Data Validation ---
        required_columns = ['BM', 'balanced_acc', 'fair_score', 'model_clean']
        if not all(col in df_results.columns for col in required_columns):
            missing = set(required_columns) - set(df_results.columns)
            raise ValueError(f"DataFrame is missing required columns: {missing}")
        
        if 'baseline' not in df_results['BM'].unique():
            raise ValueError("No 'baseline' models found in 'BM' column. Cannot generate a tradeoff curve.")

        # --- Data Preprocessing ---
        df_results['fairness'] = df_results['fair_score']
        
        # Create baseline dataframe
        baseline_df = df_results[df_results['BM'] == 'baseline'].copy()
        baseline_df = baseline_df.sort_values(by='fairness').reset_index(drop=True)
        
        # Create methods dataframe (non-baseline)
        methods_df = df_results[df_results['BM'] != 'baseline'].copy()
        
        print("Data loaded and preprocessed successfully.")
        print(f"Found {len(baseline_df)} baseline models and {len(methods_df)} mitigation methods.")
        
        return baseline_df, methods_df, df_results
        
    except FileNotFoundError:
        print(f"Error: The data file was not found at '{data_path}'. Please check the path.")
        raise
    except ValueError as e:
        print(f"Data Validation Error: {e}")
        raise
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        raise


def load_multiple_datasets(data_paths_dict):
    """
    Loads multiple datasets for multi-attribute analysis.
    
    Args:
        data_paths_dict (dict): Dictionary with keys like '1_attr', '2_attr', '3_attr'
                               and values as file paths
                               
    Returns:
        tuple: (baseline_dfs_dict, results_dfs_dict, full_dfs_dict)
    """
    baseline_dfs = {}
    results_dfs = {}
    full_dfs = {}
    
    for key, path in data_paths_dict.items():
        print(f"\nLoading {key}...")
        baseline_df, results_df, full_df = load_and_prepare_data(path)
        baseline_dfs[key] = baseline_df
        results_dfs[key] = results_df
        full_dfs[key] = full_df
    
    return baseline_dfs, results_dfs, full_dfs

data_paths = {
    'Age': '../../../experiments_intersec/wids_patient_age_agg.csv',
    'Age + Payer': '../../../experiments_intersec/wids_patient_age_payer_agg.csv',
    'Age + Payer + Race': '../../../experiments_intersec/wids_patient_age_race_payer_agg.csv'
}

baseline_dfs, results_dfs, full_dfs = load_multiple_datasets(data_paths)

# Create trend analysis plot
fig1 = analyze_multi_attribute_trends(results_dfs, baseline_dfs)
fig1.show()

# Create evolution subplots
fig2 = create_tradeoff_evolution_plot(results_dfs, baseline_dfs)
fig2.show()


Loading Age...
Data loaded and preprocessed successfully.
Found 8 baseline models and 105 mitigation methods.

Loading Age + Payer...
Data loaded and preprocessed successfully.
Found 8 baseline models and 102 mitigation methods.

Loading Age + Payer + Race...
Data loaded and preprocessed successfully.
Found 8 baseline models and 103 mitigation methods.
